# GEMM Unit Vivado Synthesis Analysis

Vivado 2022.2 합성 결과를 분석합니다.
- **Target FPGA**: Xilinx Alveo U55C (xcu55c-fsvh2892-2L-e)
- **설정 비교**: 16×16 vs 32×32 Matrix Size
- **리포트 종류**: Utilization, Timing, Power

In [ ]:
import sys
import os
sys.path.insert(0, "/home/jaeyongjang/project.local/hwexplorer")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['figure.figsize'] = (12, 5)

from hwexplorer.report_parser import ParserFactory

# Report paths
BASE_DIR = "/home/jaeyongjang/project.local/vortex/hw/syn/xilinx/dut/unittest"
CONFIGS = {
    "16x16": os.path.join(BASE_DIR, "gemm_unit_16x16"),
    "32x32": os.path.join(BASE_DIR, "gemm_unit_32x32"),
}

# Parser instances
util_parser = ParserFactory.instance().construct("vivado_utilization_parser")
timing_parser = ParserFactory.instance().construct("vivado_timing_parser")
power_parser = ParserFactory.instance().construct("vivado_power_parser")

print("Setup complete.")

## 1. Power Summary 비교

In [ ]:
# Power Summary
summary_rows = []
for cfg_name, cfg_dir in CONFIGS.items():
    s = power_parser.parseSummary(os.path.join(cfg_dir, "power.rpt"))
    s["config"] = cfg_name
    summary_rows.append(s)

df_power_summary = pd.DataFrame(summary_rows).set_index("config")
display(df_power_summary[["total_power", "dynamic_power", "static_power", "fpga_power", "hbm_power", "junction_temperature", "confidence_level"]])

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart: Power breakdown
power_cols = ["dynamic_power", "static_power"]
df_power_summary[power_cols].plot(kind="bar", stacked=True, ax=axes[0], color=["#2196F3", "#FF9800"])
axes[0].set_title("Power Breakdown (W)")
axes[0].set_ylabel("Power (W)")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.2f", padding=2, fontsize=9)

# Bar chart: Total power comparison
df_power_summary["total_power"].plot(kind="bar", ax=axes[1], color=["#4CAF50", "#F44336"])
axes[1].set_title("Total On-Chip Power (W)")
axes[1].set_ylabel("Power (W)")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].bar_label(axes[1].containers[0], fmt="%.3f", padding=3)

plt.tight_layout()
plt.show()

## 2. Utilization 비교 (Top-level + 주요 서브모듈)

In [ ]:
# Define submodules to analyze
CELL_NAMES = [
    (r"VX_gemm_unit_top$", "top"),
    (r"VX_gemm_unit_top/gemm_unit/u_mxu$", "u_mxu"),
    (r"VX_gemm_unit_top/gemm_unit/u_prealigner$", "u_prealigner"),
    (r"VX_gemm_unit_top/gemm_unit/u_merge_out_reg$", "u_merge_out_reg"),
    (r"VX_gemm_unit_top/gemm_unit/u_act_reduce$", "u_act_reduce"),
    (r"VX_gemm_unit_top/gemm_unit/gen_accumulator\[\d+\]\.u_accumulator$", "accumulator"),
    (r"VX_gemm_unit_top/gemm_unit/gen_out_scaler\[\d+\]\.u_out_scaler$", "out_scaler"),
    (r"VX_gemm_unit_top/gemm_unit/gen_int2fp\[\d+\]\.u_int2fp$", "int2fp"),
    (r"VX_gemm_unit_top/gemm_unit/gen_acc_mem\[\d+\]\.VX_sp_ram_instance$", "acc_mem"),
    (r"VX_gemm_unit_top/gemm_unit/u_pre_proc_pipe_buffer$", "u_pre_proc_pipe"),
    (r"VX_gemm_unit_top/gemm_unit/u_prealign_blk_idx_pipe$", "u_blk_idx_pipe"),
    (r"VX_gemm_unit_top/gemm_unit/u_scaler_bypass_pipe$", "u_scaler_bypass"),
    (r"VX_gemm_unit_top/gemm_unit/u_acc_rd_fifo$", "u_acc_rd_fifo"),
]

util_dfs = {}
for cfg_name, cfg_dir in CONFIGS.items():
    df = util_parser.parse(os.path.join(cfg_dir, "post_impl_util.rpt"), CELL_NAMES)
    df["config"] = cfg_name
    util_dfs[cfg_name] = df

df_util_all = pd.concat(util_dfs.values(), ignore_index=True)
display(df_util_all.pivot_table(
    index="module", columns="config",
    values=["total_luts", "ffs", "uram", "dsp_blocks", "num_of_instance"],
    aggfunc="first"
).round(0))

In [ ]:
# Utilization comparison bar chart: LUTs and FFs per submodule
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

resource_cols = ["total_luts", "ffs", "dsp_blocks"]
titles = ["Total LUTs", "Flip-Flops", "Dsp Blocks"]
colors_map = {"16x16": "#2196F3", "32x32": "#F44336"}

for ax, col, title in zip(axes, resource_cols, titles):
    submodules = df_util_all[df_util_all["module"] != "top"]["module"].unique()
    pivot = df_util_all[df_util_all["module"].isin(submodules)].pivot(index="module", columns="config", values=col)
    pivot = pivot.sort_values("16x16", ascending=True)
    pivot.plot(kind="barh", ax=ax, color=[colors_map[c] for c in pivot.columns])
    ax.set_title(title)
    ax.set_xlabel("Count")
    ax.legend(title="Config")

plt.tight_layout()
plt.show()

In [ ]:
# Top-level resource usage comparison pie charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (cfg_name, cfg_dir) in zip(axes, CONFIGS.items()):
    df = util_dfs[cfg_name]
    # Exclude 'top' and compute "other" category
    submod = df[df["module"] != "top"].copy()
    top_luts = df[df["module"] == "top"]["total_luts"].values[0]
    submod_luts = submod["total_luts"] * submod["num_of_instance"]
    other = top_luts - submod_luts.sum()

    labels = list(submod["module"]) + ["other"]
    sizes = list(submod_luts) + [max(0, other)]

    # Filter out tiny slices
    threshold = top_luts * 0.01
    filtered_labels = []
    filtered_sizes = []
    small_sum = 0
    for l, s in zip(labels, sizes):
        if s >= threshold:
            filtered_labels.append(l)
            filtered_sizes.append(s)
        else:
            small_sum += s
    if small_sum > 0:
        filtered_labels.append("others (<1%)")
        filtered_sizes.append(small_sum)

    ax.pie(filtered_sizes, labels=filtered_labels, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 8})
    ax.set_title(f"LUT Distribution — {cfg_name}")

plt.tight_layout()
plt.show()

In [ ]:
# Top-level resource usage comparison pie charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (cfg_name, cfg_dir) in zip(axes, CONFIGS.items()):
    df = util_dfs[cfg_name]
    # Exclude 'top' and compute "other" category
    submod = df[df["module"] != "top"].copy()
    top_ffs = df[df["module"] == "top"]["ffs"].values[0]
    submod_ffs = submod["ffs"] * submod["num_of_instance"]
    other = top_ffs - submod_ffs.sum()

    labels = list(submod["module"]) + ["other"]
    sizes = list(submod_ffs) + [max(0, other)]

    # Filter out tiny slices
    threshold = top_ffs * 0.01
    filtered_labels = []
    filtered_sizes = []
    small_sum = 0
    for l, s in zip(labels, sizes):
        if s >= threshold:
            filtered_labels.append(l)
            filtered_sizes.append(s)
        else:
            small_sum += s
    if small_sum > 0:
        filtered_labels.append("others (<1%)")
        filtered_sizes.append(small_sum)

    ax.pie(filtered_sizes, labels=filtered_labels, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 8})
    ax.set_title(f"ff Distribution — {cfg_name}")

plt.tight_layout()
plt.show()

## 3. Timing 분석

In [ ]:
# Timing summary comparison
timing_rows = []
for cfg_name, cfg_dir in CONFIGS.items():
    df = timing_parser.parse(os.path.join(cfg_dir, "timing.rpt"), [cfg_name])
    timing_rows.append(df.iloc[0].to_dict())

df_timing = pd.DataFrame(timing_rows).set_index("module")
display(df_timing)

# Slack visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# WNS comparison
colors = ["#4CAF50" if v >= 0 else "#F44336" for v in df_timing["wns"]]
df_timing["wns"].plot(kind="bar", ax=axes[0], color=colors)
axes[0].axhline(y=0, color='black', linewidth=0.5, linestyle='--')
axes[0].set_title("WNS (Worst Negative Slack)")
axes[0].set_ylabel("Slack (ns)")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].bar_label(axes[0].containers[0], fmt="%.3f", padding=3)

# Data path delay & logic levels
x = np.arange(len(df_timing))
w = 0.35
bars1 = axes[1].bar(x - w/2, df_timing["max_data_path_delay"], w, label="Data Path Delay (ns)", color="#2196F3")
axes[1].bar_label(bars1, fmt="%.3f", padding=3, fontsize=9)
ax2 = axes[1].twinx()
bars2 = ax2.bar(x + w/2, df_timing["max_logic_levels"], w, label="Logic Levels", color="#FF9800", alpha=0.7)
ax2.bar_label(bars2, fmt="%d", padding=3, fontsize=9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_timing.index)
axes[1].set_ylabel("Delay (ns)")
ax2.set_ylabel("Logic Levels")
axes[1].set_title("Max Data Path Delay & Logic Levels")
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Detailed timing paths: Slack distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (cfg_name, cfg_dir) in zip(axes, CONFIGS.items()):
    df_paths = timing_parser.parseTimingReport(os.path.join(cfg_dir, "timing.rpt"))
    ax.hist(df_paths["slack"], bins=30, color="#2196F3" if cfg_name == "16x16" else "#F44336", alpha=0.8, edgecolor="white")
    ax.axvline(x=0, color='red', linewidth=1, linestyle='--', label='Zero Slack')
    ax.set_title(f"Slack Distribution — {cfg_name}")
    ax.set_xlabel("Slack (ns)")
    ax.set_ylabel("# Paths")
    met = (df_paths["slack"] >= 0).sum()
    violated = (df_paths["slack"] < 0).sum()
    ax.legend([f"MET: {met}, VIOLATED: {violated}"], fontsize=9)

plt.tight_layout()
plt.show()

## 4. 계층별 Power 분석

In [ ]:
# Hierarchical power comparison
POWER_CELLS = [
    (r"VX_gemm_unit_top/gemm_unit/u_mxu$", "u_mxu"),
    (r"VX_gemm_unit_top/gemm_unit/u_prealigner$", "u_prealigner"),
    (r"VX_gemm_unit_top/gemm_unit/u_merge_out_reg$", "u_merge_out_reg"),
    (r"VX_gemm_unit_top/gemm_unit/u_act_reduce$", "u_act_reduce"),
    (r"VX_gemm_unit_top/gemm_unit/gen_accumulator\[\d+\]\.u_accumulator$", "accumulator"),
    (r"VX_gemm_unit_top/gemm_unit/gen_out_scaler\[\d+\]\.u_out_scaler$", "out_scaler"),
    (r"VX_gemm_unit_top/gemm_unit/gen_int2fp\[\d+\]\.u_int2fp$", "int2fp"),
    (r"VX_gemm_unit_top/gemm_unit/gen_acc_mem\[\d+\]\.VX_sp_ram_instance$", "acc_mem"),
    (r"VX_gemm_unit_top/gemm_unit/u_pre_proc_pipe_buffer$", "u_pre_proc_pipe"),
    (r"VX_gemm_unit_top/gemm_unit/u_prealign_blk_idx_pipe$", "u_blk_idx_pipe"),
    (r"VX_gemm_unit_top/gemm_unit/u_scaler_bypass_pipe$", "u_scaler_bypass"),
    (r"VX_gemm_unit_top/gemm_unit/u_acc_rd_fifo$", "u_acc_rd_fifo"),
    (r"VX_gemm_unit_top/gemm_unit/gen_mxu_output_dly\[\d+\]\.u_mxu_output_dly_pipe$", "mxu_output_dly"),
    (r"VX_gemm_unit_top/gemm_unit/u_zp_mul_out_reg$", "u_zp_mul_out_reg"),
    (r"VX_gemm_unit_top/gemm_unit/u_in_pipe$", "u_in_pipe"),
]

power_dfs = {}
for cfg_name, cfg_dir in CONFIGS.items():
    df = power_parser.parse(os.path.join(cfg_dir, "power.rpt"), POWER_CELLS)
    # Multiply power by num_of_instance to get total power
    df["total_power"] = df["power"] * df["num_of_instance"]
    df["config"] = cfg_name
    power_dfs[cfg_name] = df

df_power_all = pd.concat(power_dfs.values(), ignore_index=True)
pivot_power = df_power_all.pivot(index="module", columns="config", values="total_power")
display(pivot_power.sort_values("16x16", ascending=False).round(3))

In [ ]:
# Power breakdown bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (cfg_name, cfg_dir) in zip(axes, CONFIGS.items()):
    power_dfs[cfg_name]["all_inst_total_power"] = power_dfs[cfg_name]["total_power"] * power_dfs[cfg_name]["num_of_instance"]
    df = power_dfs[cfg_name].sort_values("all_inst_total_power", ascending=True)
    colors = plt.cm.tab20(np.linspace(0, 1, len(df)))
    ax.barh(df["module"], df["all_inst_total_power"], color=colors)
    ax.set_title(f"Power by Sub-module — {cfg_name}")
    ax.set_xlabel("Total Power (W)")
    for i, (v, n) in enumerate(zip(df["all_inst_total_power"], df["num_of_instance"])):
        ax.text(v + 0.005, i, f"{v:.3f}W (×{n})", va='center', fontsize=7)

plt.tight_layout()
plt.show()

## 5. On-Chip Components Power

In [ ]:
# On-Chip Components power comparison
onchip_dfs = {}
for cfg_name, cfg_dir in CONFIGS.items():
    df = power_parser.parseOnChipComponents(os.path.join(cfg_dir, "power.rpt"))
    df["config"] = cfg_name
    onchip_dfs[cfg_name] = df

df_onchip = pd.concat(onchip_dfs.values(), ignore_index=True)
# Filter only rows with non-zero power
df_onchip_valid = df_onchip[df_onchip["power"] > 0.0].copy()

pivot_onchip = df_onchip_valid.pivot(index="component", columns="config", values="power").fillna(0)
display(pivot_onchip)

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
pivot_onchip_sorted = pivot_onchip.sort_values("16x16", ascending=True)
pivot_onchip_sorted.plot(kind="barh", ax=ax, color=["#2196F3", "#F44336"])
ax.set_title("On-Chip Component Power (W)")
ax.set_xlabel("Power (W)")
ax.legend(title="Config")
plt.tight_layout()
plt.show()

## 6. 종합 비교 (16×16 vs 32×32)

In [ ]:
# Comprehensive comparison table
summary_data = []
for cfg_name, cfg_dir in CONFIGS.items():
    # Utilization
    util_df = util_dfs[cfg_name]
    top = util_df[util_df["module"] == "top"].iloc[0]
    # Timing
    timing_df = timing_parser.parse(os.path.join(cfg_dir, "timing.rpt"), [cfg_name])
    t = timing_df.iloc[0]
    # Power
    ps = power_parser.parseSummary(os.path.join(cfg_dir, "power.rpt"))

    summary_data.append({
        "Config": cfg_name,
        "Total LUTs": int(top["total_luts"]),
        "Logic LUTs": int(top["logic_luts"]),
        "FFs": int(top["ffs"]),
        "URAM": int(top["uram"]),
        "DSP": int(top["dsp_blocks"]),
        "WNS (ns)": t["wns"],
        "Timing Met?": "Yes" if t["wns"] >= 0 else "NO",
        "Clock Period (ns)": t["requirement"],
        "Fmax (MHz)": round(1000 / (t["requirement"] - min(t["wns"], 0)), 1),
        "Max Logic Levels": int(t["max_logic_levels"]),
        "Total Power (W)": ps.get("total_power", 0),
        "Dynamic (W)": ps.get("dynamic_power", 0),
        "Static (W)": ps.get("static_power", 0),
        "Tj (°C)": ps.get("junction_temperature", 0),
    })

df_summary = pd.DataFrame(summary_data).set_index("Config")
display(df_summary.T)

# Scaling ratio
print(f"\n=== 32x32 / 16x16 Scaling Ratio ===")
for col in ["Total LUTs", "FFs", "URAM", "DSP", "Total Power (W)", "Dynamic (W)"]:
    v16 = df_summary.loc["16x16", col]
    v32 = df_summary.loc["32x32", col]
    ratio = v32 / v16 if v16 != 0 else float('inf')
    print(f"  {col}: {ratio:.2f}x  ({v16} → {v32})")